# 4 · Running the full pipeline

This notebook shows the real end-to-end run: MSWX climate, SoilGrids soil
(+ optional PTF), NPK, cropland masking and the SIMPLACE exporters.

> **Data required.** Unlike notebooks 1–3, this one reads the input
> datasets configured under `paths:` and `reference:` in `config.yaml`.
> The cells are marked *read-only* here; run them where those paths are
> reachable.

## The moving parts

The [`Pipeline`](../api/pipeline.md#data4simplace.pipeline.Pipeline) runs
each stage only when its flag is set, keeps intermediate `xarray.Dataset`s
in memory, and hands them to the exporters:

| Stage | Class | Output |
| --- | --- | --- |
| Climate | `MSWXHandler` | gridded daily climate |
| Soil | `SoilGridsHandler` (+ `saxton_rawls`) | gridded soil + hydraulics |
| NPK | `NPKHandler` | gridded nutrients |
| Mask | `CroplandMask` | agricultural cells only |
| Export | `WeatherExporter` / `SoilExporter` / `ManagementExporter` | SIMPLACE CSVs |

## Load the project configuration

In [2]:
from data4simplace import load_config

config = load_config('../../config.yaml')  # repo-root config.yaml
enabled = [n for n, on in config.flags.model_dump().items() if on]
print('Enabled stages:', ', '.join(enabled))
print('Output dir     :', config.paths.output_dir)

Enabled stages: run_climate_processing, run_soil_processing, run_npk_processing, apply_agricultural_mask, export_simplace_weather, export_simplace_soil, export_simplace_management
Output dir     : output


## Run it

```python
from data4simplace.pipeline import Pipeline

result = Pipeline(config).run()
print(f'Wrote {len(result.written)} file(s):')
for path in result.written:
    print(' -', path)
```

The call returns a
[`PipelineResult`](../api/pipeline.md#data4simplace.pipeline.PipelineResult)
holding the in-memory `climate`, `soil`, `hydraulic`, `npk` datasets, the
`cell_table`, and the list of `written` output paths.

## Driving individual stages

You do not have to run the whole pipeline. Each handler takes the config
and exposes a `.load()` returning an `xarray.Dataset`, so you can inspect
one dataset at a time:

```python
from data4simplace.climate import MSWXHandler
from data4simplace.soil import SoilGridsHandler
from data4simplace.grid import TargetGrid

grid = TargetGrid.from_config(config.grid)
climate = MSWXHandler(config).load()     # gridded daily climate
soil = SoilGridsHandler(config).load()   # gridded soil layers
cells = grid.cell_table()                # SimplaceID lookup
```

In [3]:
from data4simplace.climate import MSWXHandler
from data4simplace.soil import SoilGridsHandler
from data4simplace.grid import TargetGrid

grid = TargetGrid.from_config(config.grid)
climate = MSWXHandler(config).load()     # gridded daily climate
soil = SoilGridsHandler(config).load()   # gridded soil layers
cells = grid.cell_table()                # SimplaceID lookup

/home/muduchuru/miniforge3/envs/sdba/lib/python3.10/site-packages/xarray/core/dataset.py:277: UserWarning: The specified chunks separate the stored chunks along dimension "lat" starting at index 512. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(
/home/muduchuru/miniforge3/envs/sdba/lib/python3.10/site-packages/xarray/core/dataset.py:277: UserWarning: The specified chunks separate the stored chunks along dimension "lon" starting at index 512. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(
No MSWX files for HURS in requested window
/home/muduchuru/miniforge3/envs/sdba/lib/python3.10/site-packages/dask/array/core.py:4888: PerformanceWarning: Increasing number of chunks by factor of 36
  result = blockwise(
/home/muduchuru/miniforge3/envs/sdba/lib/python3.10/site-packages/dask/array/core.py:4888: PerformanceWarning: Increasing number of chunks by factor of 36
  result = blockwise(
No soilgrids_root

RuntimeError: No SoilGrids layers could be loaded. Set paths.soilgrids_root to a directory of GeoTIFF coverages or pre-fetch via fetch_wcs.

## Exporting on their own

The exporters take a processed dataset, the `cell_table` and an output
directory. They inspect the SIMPLACE reference files to reproduce the
exact delimiter, headers, `-99` sentinel and depth horizons:

```python
from data4simplace.exporters import WeatherExporter

exporter = WeatherExporter(config, config.reference.weather_dir)
written = exporter.export(climate, cells, config.paths.output_dir)
```

## Or just use the CLI

```bash
# validate + show the plan
data4simplace --config config.yaml --dry-run

# full run with debug logging
data4simplace --config config.yaml --verbose
```

See the [command-line reference](../cli.md) for options and exit codes.